# Appendix B — normalisation and the depth artefact

**Optional. Nothing in the workshop depends on this notebook.**

Notebook 03 normalises with median scaling and `log1p`, mentions in passing that this
does not fully remove sequencing depth, and moves on. That is the right level of
detail for a first pass, and median scaling is what you will meet in most other
people's code.

This appendix is the argument underneath it:

- four ways to normalise a targeted panel, and how each one fails
- why area-normalisation helps for some artefacts and makes others worse
- Pearson residuals, and how to use a representation that is not an expression matrix
- a diagnostic for whether a principal component is measuring depth rather than biology
- an explorer that runs six remedies and reports what each one costs

**This dataset has a genuine depth artefact**, so the explorer is a live demonstration
rather than a hypothetical. Worth an hour before you analyse your own data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=90, frameon=False)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
NAVY, GOLD, CORAL, ICE = "#001158", "#FBAE40", "#F26B43", "#BCD2FF"

adata = sc.read_h5ad(DATA / "ovarian_qc.h5ad")   # from notebook 02
print(adata)

## 1. Normalisation

A cell's total count here is a mixture of at least four things:

1. how transcriptionally active the cell actually is
2. how big the polygon somebody drew around it is
3. how much signal leaked in from its neighbours
4. how well that part of the section was fixed, permeabilised and focused

Number 2 is new relative to scRNA-seq, and it is *measured* — `cell_area` is sitting
right there in `obs`, and it correlates strongly with `total_counts`. So the question
is not only how to scale, but **which of these you want to keep**. Cell size is partly
biology.

Four defensible options, each wrong in its own way:

| Option | Assumes | Fails when |
|---|---|---|
| **Median-scaled counts** (`normalize_total`) | total count tracks activity | compared populations differ in size |
| **Fixed target sum** (`target_sum=100`) | comparability across datasets matters most | counts are low — scaling up inflates variance |
| **Divide by `cell_area`** | the measurement is a density | the polygon is wrong; the error goes into the denominator |
| **Model counts directly** (Pearson residuals, GLM-PCA, NB) | you would rather not scale at all | it is less familiar to your readers and reviewers |

There is no consensus. We use median-scaled + `log1p` below because it is what you
will meet in other people's code, and we keep the area-normalised alternative in a
layer so exercise 3.1 can compare them.

> **Worth knowing:** `sc.experimental.pp.normalize_pearson_residuals` handles low
> counts better than `log1p` and sidesteps the scaling question entirely. It is
> increasingly used for imaging-based data. If you are comfortable, try swapping it
> in and see whether your clusters change.

Three practical rules, whichever you choose:

- **Keep raw counts in a layer.** Normalise, scale, cluster — then put the
  log-normalised values back before any marker or expression plot. Reporting numbers
  off a z-scaled matrix is a common and invisible error.
- **Take the control features out first** (we did, in notebook 02) — otherwise
  `total_counts` includes background and you normalise by it.
- **State it, with the number.** "Median-scaled, log1p" is a methods sentence.
  "Standard normalisation" is not.

In [ ]:
import scipy.sparse as sp

adata.layers["counts"] = adata.X.copy()


def normalise(counts, method, obs=None, log=True):
    """Return a normalised (and optionally log1p'd) copy of a count matrix.

    One function for all three scaling schemes, so the only thing that differs
    between them is the size factor. That is the whole point: they are the same
    operation with a different denominator.
    """
    X = sp.csr_matrix(counts, dtype="float32", copy=True)
    total = np.asarray(X.sum(axis=1)).ravel()

    if method == "median":
        # scanpy's normalize_total default: scale every cell to the median count
        sf = total / np.median(total[total > 0])
    elif method == "target":
        # fixed target sum, comparable across datasets
        sf = total / 100.0
    elif method == "area":
        # transcripts per unit area, rescaled to the median cell so the numbers
        # stay on a familiar scale
        area = obs["cell_area"].to_numpy(dtype="float64")
        sf = area / np.median(area)
    else:
        raise ValueError(method)

    sf[sf <= 0] = 1.0
    X = sp.diags(1.0 / sf) @ X
    X = sp.csr_matrix(X)
    if log:
        X.data = np.log1p(X.data)      # log1p(0) == 0, so sparsity is preserved
    return X


for name, method in [("lognorm", "median"),
                     ("lognorm_target100", "target"),
                     ("lognorm_area", "area")]:
    adata.layers[name] = normalise(adata.layers["counts"], method, adata.obs)
    print(f"{name:<20} built")

# carry the median-scaled version forward as the working matrix
adata.X = adata.layers["lognorm"].copy()
adata.uns["log1p"] = {"base": None}     # so scanpy knows X is already logged

### Now look at what you just did

The schemes differ in one number per cell — the size factor. The fastest way to see
what that number is doing is to plot expression against **cell area**, which is the
variable they disagree about.

No clustering needed, so this is cheap and it makes the mechanism obvious.

In [ ]:
def area_profile(layer, genes, n_bins=10):
    """Mean normalised expression per cell-area decile."""
    q = pd.qcut(adata.obs["cell_area"], n_bins, labels=False, duplicates="drop")
    X = adata[:, genes].layers[layer]
    X = X.toarray() if sp.issparse(X) else np.asarray(X)
    out = np.vstack([X[q == b].mean(axis=0) for b in range(q.max() + 1)])
    centres = [adata.obs.loc[q == b, "cell_area"].median() for b in range(q.max() + 1)]
    return np.asarray(centres), out


# pick genes that are abundant enough for the means to be stable
abundance = pd.Series(np.asarray(adata.layers["counts"].sum(axis=0)).ravel(),
                      index=adata.var_names).sort_values(ascending=False)
genes = abundance.index[:3].tolist()
print("plotting:", genes)

schemes = [("counts", "raw counts", "0.45"),
           ("lognorm", "median-scaled", NAVY),
           ("lognorm_target100", "target sum 100", GOLD),
           ("lognorm_area", "per unit area", CORAL)]

fig, axes = plt.subplots(1, len(genes), figsize=(4.6 * len(genes), 3.8))
axes = np.atleast_1d(axes)
for k, (g, ax) in enumerate(zip(genes, axes)):
    for layer, label, colour in schemes:
        centres, prof = area_profile(layer, genes)
        v = prof[:, k]
        ax.plot(centres, v / max(v.mean(), 1e-9), "o-", color=colour, label=label, ms=4)
    ax.set_xlabel("cell area (µm²)")
    ax.set_title(g)
axes[0].set_ylabel("mean expression\n(scaled to its own mean)")
axes[-1].legend(frameon=False, fontsize=8)
sns.despine(); plt.tight_layout(); plt.show()

**Read the slopes.**

- **Raw counts** climb with area — bigger polygon, more molecules. Some of that is a
  genuinely larger, more active cell; some is just a larger box.
- **Median-scaled** flattens it partially: it divides by *total counts*, which is
  itself correlated with area, so the size effect is reduced but not removed.
- **Per unit area** flattens it most, because area is exactly what you divided by.
  Notice it can now *invert*: a large cell with ordinary activity looks dilute.
- **Target sum 100** tracks median-scaling in shape but with different variance.

None of these lines is the correct one. They encode different definitions of "how
much is this cell expressing", and the right definition depends on your question.

In [ ]:
# The same point as one number: how much cell-size signal survives each scheme?
from scipy.stats import spearmanr

print("Spearman correlation between a cell's normalised total and its area\n")
for layer, label, _ in schemes:
    X = adata.layers[layer]
    tot = np.asarray(X.sum(axis=1)).ravel()
    rho = spearmanr(tot, adata.obs["cell_area"]).statistic
    print(f"  {label:<18} rho = {rho:+.3f}")

### Why is none of them zero?

You probably expected the area-normalised row to come out near zero and it did not.
Two reasons, both worth understanding:

1. **`log1p` is non-linear.** You divided by a size factor, then logged. The row sums
   *after* logging are no longer the quantity you controlled.
2. **Detection rate carries size.** A bigger cell has more molecules, so it detects
   more distinct genes. No size factor removes that — dividing by 2 turns a count of
   4 into 2, but it never turns a zero into a non-zero.

Point 2 is the one to remember. **Detection rate is the part of cell size that
survives every normalisation scheme**, which is why a cluster of "just big cells" or
"just small cells" can appear no matter what you do. When you meet one in section 2,
this is usually the reason.

So the choice is not between removing size and keeping it — it is about *how much*
you remove, and whether that serves your question:

- asking about **cell state** (is this fibroblast activated?) → size is mostly nuisance
- asking about **tissue output** (how much collagen is this region producing?) → size
  is the biology, and you probably want it back

### The fourth option: don't scale at all

Instead of dividing, model the counts. Analytic **Pearson residuals** ask: how far is
this count from what you would expect given the cell's depth and the gene's
abundance, in units of its own standard deviation?

$$z_{ij} = \\frac{x_{ij} - \\mu_{ij}}{\\sqrt{\\mu_{ij} + \\mu_{ij}^2/\\theta}}
\\qquad \\mu_{ij} = \\frac{\\sum_j x_{ij} \\; \\sum_i x_{ij}}{\\sum_{ij} x_{ij}}$$

This handles low counts better than `log1p` and sidesteps the size-factor question
altogether. The catch is memory: residuals are **dense**, so we compute them on the
most abundant genes only. `sc.experimental.pp.normalize_pearson_residuals` does the
same thing if you have the RAM for the full matrix.

In [ ]:
def pearson_residual_pca(adata, n_genes=1000, n_comps=50, theta=100.0):
    """Analytic Pearson residuals on the top-expressed genes, then PCA.

    Written out rather than called from a library so you can see there is no
    magic here — it is one formula.
    """
    from sklearn.utils.extmath import randomized_svd

    keep = abundance.index[:n_genes]
    X = adata[:, keep].layers["counts"]
    X = X.toarray().astype("float32") if sp.issparse(X) else np.asarray(X, "float32")

    row = X.sum(axis=1, keepdims=True)
    col = X.sum(axis=0, keepdims=True)
    mu = row @ col / X.sum()
    z = (X - mu) / np.sqrt(mu + mu**2 / theta)
    z = np.clip(z, -np.sqrt(X.shape[0]), np.sqrt(X.shape[0]))   # clip as in Lause et al.

    z -= z.mean(axis=0, keepdims=True)
    # randomised SVD: we only want 50 components, and a full SVD of a
    # 40,000 x 1,000 matrix takes minutes for no benefit
    u, s, _ = randomized_svd(z, n_components=n_comps, random_state=0)
    return (u * s).astype("float32")


# ~1000 genes x n_cells as float32. Skip this cell if your laptop is struggling —
# nothing later in the notebook depends on it.
RUN_PEARSON = True
if RUN_PEARSON:
    adata.obsm["X_pca_pearson"] = pearson_residual_pca(adata)
    print("stored obsm['X_pca_pearson']", adata.obsm["X_pca_pearson"].shape)
else:
    print("skipped")

### Using the residuals downstream

This is the part that trips people up. Pearson residuals give you a **representation**,
not a normalised expression matrix, and the two are used in different places:

| Task | Use |
|---|---|
| neighbours, clustering, UMAP | `obsm["X_pca_pearson"]` |
| marker detection, dotplots, expression plots, gene scores | `layers["lognorm"]` |

**Do not put residuals into `adata.X`.** They are dense, signed, and centred, so
`rank_genes_groups` produces meaningless log-fold-changes, `score_genes` breaks, and
every dotplot colour scale becomes uninterpretable. A residual of −1.8 is not an
expression level.

The standard recipe is therefore: **cluster on residuals, then describe the clusters
with log-normalised counts.** The cell below sets a single switch that the clustering
in section 2 obeys.

> **The scanpy equivalent.** `sc.experimental.pp.normalize_pearson_residuals_pca`
> does all of the above in one call and writes `obsm["X_pca"]`. It uses the full
> gene set, so it needs considerably more memory than our 1000-gene version. If it
> runs on your machine, prefer it — and note that it overwrites `X_pca`, so the two
> paths are then no longer comparable side by side.


In [ ]:
# Which representation should the neighbour graph be built from?
#   "pca"     — PCA of scaled log-normalised counts (the default path)
#   "pearson" — PCA of analytic Pearson residuals (needs RUN_PEARSON above)
REPRESENTATION = "pca"

if REPRESENTATION == "pearson" and "X_pca_pearson" not in adata.obsm:
    raise RuntimeError("run the Pearson cell above with RUN_PEARSON = True first")
print("clustering will use:", REPRESENTATION)

### Which one do we carry forward?

**Median-scaled + `log1p`**, in `adata.X` and in `layers["lognorm"]`. Not because it
is best, but because it is what you will meet in other people's code and in the
scanpy documentation, and because a workshop should teach the default before it
teaches the alternatives.

The other three stay in `layers` and `obsm` so exercise 3.1 can compare them.

> **Keep raw counts in a layer, always.** We normalise, scale and cluster below —
> then put the log-normalised values back into `adata.X` before any marker or
> expression plot. Reporting numbers off a z-scaled matrix is a common, invisible
> and quite serious error.

> **No HVG selection.** In scRNA-seq you pick ~2000 variable genes out of 20,000 to
> denoise the distance metric. Here the panel is already a curated 5,000 genes chosen
> to discriminate cell types. Running `highly_variable_genes` on top mostly selects
> the most *abundant* genes and throws away the rare-but-informative markers the
> panel was designed around. Skip it. If you must reduce, do it on biological
> grounds, not variance.

> **Use the detection flag from notebook 02.** `adata.var["above_background"]` marks
> the genes that beat the negative-control null. Cluster on everything — a weak gene
> contributes little either way — but when you read markers off
> `rank_genes_groups`, check the flag before believing a gene you have not seen
> before. A "marker" that never rose above background is not a marker.


### Is a PC just measuring depth?

Colouring the UMAP is the visual check. This is the numeric one, and it is more
useful because it tells you *which* component is the problem and *what kind* of
problem it is.

Read three things off the output:

**1. Which PCs are technical.** A correlation above about 0.6 with a QC metric means
that component is largely measuring the metric, not biology.

**2. Detection rate or cell size?** These are different problems.
- correlates with `total_counts` and `n_genes_by_counts`, but *not* `cell_area` →
  a **detection-rate** axis. Some cells simply yielded more molecules. Area
  normalisation will not help, because area is not what varies.
- correlates with `cell_area` → a **size** axis, where the polygons differ. Here
  area-normalisation is the natural fix.

**3. How flat is the spectrum.** A targeted panel at a few hundred counts per cell
gives a much flatter PCA spectrum than scRNA-seq — PC1 explaining 1–3% rather than
15–25% is normal, not a bug. Simulating a panel at 120 counts per cell gives PC1
around 3%; the same panel at 3,000 counts gives 19%. A flat spectrum means no single
axis dominates, so you need more components, and dropping one costs less than usual.

In [ ]:
qc_cols = ["total_counts", "n_genes_by_counts", "cell_area"]
pcs = adata.obsm["X_pca"]
vr = adata.uns["pca"]["variance_ratio"]

rows = []
for i in range(6):
    row = {"PC": f"PC{i + 1}", "var %": round(100 * vr[i], 2)}
    for col in qc_cols:
        row[col] = round(float(np.corrcoef(pcs[:, i], adata.obs[col])[0, 1]), 2)
    rows.append(row)
tab = pd.DataFrame(rows).set_index("PC")
display(tab)

print(f"first 30 PCs capture {100 * vr[:30].sum():.1f}% of the variance, "
      f"first 50 {100 * vr[:min(50, len(vr))].sum():.1f}%")

# Which kind of technical axis is each PC? Detection and size are different
# problems with different fixes, so it is worth separating them.
DETECTION = ["total_counts", "n_genes_by_counts"]
for i in range(4):
    det = max(abs(tab.iloc[i][c]) for c in DETECTION)
    size = abs(tab.iloc[i]["cell_area"])
    if max(det, size) < 0.3:
        verdict = "looks like biology"
    elif det > size + 0.15:
        verdict = "DETECTION-RATE axis — area-normalising will not fix this"
    elif size > det + 0.15:
        verdict = "CELL-SIZE axis — area-normalisation is the natural fix"
    else:
        verdict = "mixed depth and size"
    print(f"  PC{i + 1}: {verdict}")

> **Try it yourself — how many PCs?**
>
> There is rarely a clean elbow with a targeted panel. Rather than squinting at one,
> ask the question that matters: does your smallest cluster survive?

In [ ]:
TRY_N_PCS = 30        # <-- CHANGE THIS (try 10, 30, 50)

sc.pp.neighbors(adata, n_neighbors=15, n_pcs=TRY_N_PCS, key_added="try_pcs")
sc.tl.leiden(adata, resolution=1.0, key_added="leiden_pcs", neighbors_key="try_pcs",
             flavor="igraph", n_iterations=2)
sizes = adata.obs["leiden_pcs"].value_counts()
cum = adata.uns["pca"]["variance_ratio"][:TRY_N_PCS].sum()
print(f"{TRY_N_PCS} PCs capture {100 * cum:.1f}% of the variance")
print(f"{len(sizes)} clusters; smallest has {sizes.iloc[-1]:,} cells")

### Your data has this problem. Now fix it.

If the table above shows a PC correlating strongly with `total_counts`, depth is
sitting in your embedding and every downstream step inherits it. There are five
things you can do about it, and they are not equally good:

| Fix | What it does | Cost |
|---|---|---|
| **Area-normalise** | divides by `cell_area` instead of total counts | puts segmentation error in the denominator |
| **Fixed target sum** | scales every cell to the same total | inflates variance in sparse cells |
| **Pearson residuals** | models counts instead of scaling them | dense, less familiar to reviewers |
| **Regress out depth** | fits and subtracts `total_counts` per gene | removes real biology if size *is* the signal; slow |
| **Drop PC1** | discards the offending component | crude; PC1 usually carries biology too |
| **Filter harder** | removes the low-count cells driving the gradient | deletes cells, often a whole population |

The next cell runs all of them on a subsample and reports how much depth survives, so
you can choose on evidence rather than taste.

In [ ]:
# Comparison runs on a subsample so it finishes in a couple of minutes.
# Reduce N_SUB if your laptop is slow; raise it if you want more confidence.
N_SUB = 5000            # <-- CHANGE THIS if needed

rng = np.random.default_rng(0)
idx = rng.choice(adata.n_obs, size=min(N_SUB, adata.n_obs), replace=False)
sub = adata[idx].copy()
print(f"comparing on {sub.n_obs:,} cells")

QC = ["total_counts", "cell_area"]


def depth_in_embedding(emb, obs, n_pcs=10):
    """How much of the embedding is explained by depth or size?

    Returns the largest |correlation| between any of the first n_pcs components
    and any QC metric — one number, so the fixes can be ranked.
    """
    worst, where = 0.0, ""
    for i in range(min(n_pcs, emb.shape[1])):
        for col in QC:
            r = abs(float(np.corrcoef(emb[:, i], obs[col])[0, 1]))
            if r > worst:
                worst, where = r, f"PC{i + 1} vs {col}"
    return worst, where


def embed(ad, layer, scale=True, drop_first=0, regress=False, n_comps=20):
    """Scale -> PCA on one layer, optionally regressing out depth first."""
    b = ad.copy()
    b.X = b.layers[layer].copy()
    if regress:
        sc.pp.regress_out(b, ["total_counts"])
    if scale:
        sc.pp.scale(b, max_value=10)
    sc.pp.pca(b, n_comps=n_comps, svd_solver="arpack")
    return b.obsm["X_pca"][:, drop_first:]

In [ ]:
import time

from sklearn.neighbors import NearestNeighbors

results, baseline_emb = [], None


def knn_overlap(A, B, k=30):
    """Mean fraction of each cell's k nearest neighbours shared by two embeddings.

    A measure of how much a fix rearranged the data — 1.0 means nothing moved,
    near 0 means a completely different embedding. It says nothing about which
    one is better; that is what the tissue plot below is for.
    """
    kn = lambda M: NearestNeighbors(n_neighbors=k + 1).fit(M).kneighbors(
        M, return_distance=False)[:, 1:]
    ia, ib = kn(A), kn(B)
    return float(np.mean([len(set(a) & set(b)) / k for a, b in zip(ia, ib)]))


def try_fix(name, fn, is_baseline=False):
    global baseline_emb
    t0 = time.time()
    try:
        emb = fn()
        worst, where = depth_in_embedding(emb, sub.obs)
        if is_baseline:
            baseline_emb = emb
            overlap = 1.0
        else:
            overlap = knn_overlap(baseline_emb, emb)
        results.append({"fix": name,
                        "max |r| with QC": round(worst, 3),
                        "worst pair": where,
                        "kNN overlap": round(overlap, 2),
                        "s": round(time.time() - t0, 1)})
        print(f"  {name:<26} |r| = {worst:.3f}   overlap = {overlap:.2f}   ({where})")
    except Exception as exc:                      # noqa: BLE001
        print(f"  {name:<26} failed: {type(exc).__name__}: {exc}")


print("running fixes — the regression one is by far the slowest\n")

try_fix("0. nothing (baseline)", lambda: embed(sub, "lognorm"), is_baseline=True)
try_fix("1. area-normalised", lambda: embed(sub, "lognorm_area"))
try_fix("2. target sum 100", lambda: embed(sub, "lognorm_target100"))
try_fix("3. drop PC1", lambda: embed(sub, "lognorm", drop_first=1))
try_fix("4. regress out depth", lambda: embed(sub, "lognorm", regress=True))

if "X_pca_pearson" in adata.obsm:
    try_fix("5. Pearson residuals", lambda: adata.obsm["X_pca_pearson"][idx][:, :20])
else:
    print("  5. Pearson residuals      skipped (run the RUN_PEARSON cell above)")

display(pd.DataFrame(results).set_index("fix"))

### Reading the table

Two columns, and neither is a score you maximise.

- **`max |r| with QC`** — how much depth or size is left. Lower is better.
- **`kNN overlap`** — how much the fix rearranged the data relative to doing nothing.
  1.0 means nothing moved; near 0 means a completely different embedding.

**A low overlap is not automatically bad.** Pearson residuals produce a very
different embedding *by construction*, and that is the point of them. What the
overlap tells you is how much is at stake in the choice: if a fix drops `|r|` from
0.7 to 0.05 and barely moves the neighbourhoods, it is nearly free. If it drops `|r|`
and reshuffles everything, you have made a substantive decision and need to check it.

From testing these on simulated data with a planted artefact:

- **Pearson residuals** removed technical depth most effectively (|r| 0.69 → 0.05)
  while preserving the true group structure. They also rearranged the embedding the
  most, so the number looks alarming and is not.
- **Regressing out depth** worked (|r| → 0.20) but is the most likely to over-correct,
  because it removes cell size, which here is partly biology.
- **Area-normalisation** helped when the artefact really was polygon size, and made
  things **worse** when the depth variation was technical. Do not assume; check.
- **Dropping PC1** did not fix the correlation — it simply reappeared on PC2 — and
  cost structure. It is in the table so you can see that for yourself.

### So how do you decide?

Neither column answers "which clustering is right", and no purely numerical criterion
can. But you have something a scRNA-seq analysis does not:

> **Cluster under two schemes, then plot both on the tissue. The one that looks more
> like histology is the better one.**

That is an external validator, and it is the reason this workshop keeps returning to
the spatial plot. Use it here.

> **What if nothing gets you below ~0.4–0.5?** Then depth is genuinely confounded
> with cell type — small cells really are a different population, and no
> normalisation can separate "this cell is small" from "this cell is a lymphocyte".
> That is a finding about your tissue, not a failure of method. Report it, and show
> your conclusion holds either way.

In [ ]:
# Before discarding PC1, find out what it actually is.
loadings = pd.Series(adata.varm["PCs"][:, 0], index=adata.var_names)
print("genes with the most positive PC1 loading:")
print(loadings.nlargest(10).round(3).to_string())
print("\ngenes with the most negative PC1 loading:")
print(loadings.nsmallest(10).round(3).to_string())
print("\nIf these two lists look like two cell types, PC1 is biology and dropping")
print("it would be a mistake — fix the normalisation instead.")

> **Try it yourself — adopt a fix**
>
> Pick the one you were most convinced by and re-run the clustering with it. Then go
> back and look at the UMAP coloured by `total_counts`: the gradient should be gone,
> or much weaker.

In [ ]:
FIX = "area"        # <-- CHANGE THIS: "none", "area", "target100", "pearson", "regress"

if FIX == "none":
    adata.X = adata.layers["lognorm"].copy()
    sc.pp.scale(adata, max_value=10); sc.pp.pca(adata, n_comps=50, svd_solver="arpack")
    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
elif FIX in ("area", "target100"):
    adata.X = adata.layers[f"lognorm_{'area' if FIX == 'area' else 'target100'}"].copy()
    sc.pp.scale(adata, max_value=10); sc.pp.pca(adata, n_comps=50, svd_solver="arpack")
    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
elif FIX == "pearson":
    if "X_pca_pearson" not in adata.obsm:
        raise RuntimeError("run the Pearson cell above with RUN_PEARSON = True first")
    sc.pp.neighbors(adata, n_neighbors=15, use_rep="X_pca_pearson", n_pcs=30)
elif FIX == "regress":
    adata.X = adata.layers["lognorm"].copy()
    sc.pp.regress_out(adata, ["total_counts"])       # slow on the full object
    sc.pp.scale(adata, max_value=10); sc.pp.pca(adata, n_comps=50, svd_solver="arpack")
    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
else:
    raise ValueError(f"unknown FIX: {FIX!r}")

sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=1.0, key_added="leiden", flavor="igraph", n_iterations=2)

# whatever we clustered on, expression values go back before any marker work
adata.X = adata.layers["lognorm"].copy()
print(f"FIX = {FIX!r}: {adata.obs['leiden'].nunique()} clusters")

if FIX != "pearson":
    worst, where = depth_in_embedding(adata.obsm["X_pca"], adata.obs)
    print(f"depth left in the embedding: max |r| = {worst:.3f} ({where})")

In [ ]:
sc.pl.umap(adata, color=["leiden", "total_counts", "cell_area"],
           wspace=0.35, ncols=3, size=6)

### The external check: does it look like tissue?

Now use the thing scRNA-seq cannot. Below, the clustering before and after your fix,
side by side on the section. Judge them as an anatomist:

- Are tumour nests compact, with edges?
- Do stromal bands run as bands, or dissolve into speckle?
- Are vessels thin lines rather than scattered dots?

The clustering that produces a map you could describe to a pathologist is the better
one, whatever the two numbers said.

In [ ]:
if "leiden_before_fix" not in adata.obs:
    print("No 'before' clustering stored. Re-run the baseline clustering cell first,")
    print("then this comparison becomes available.")
else:
    xs, ys = adata.obsm["spatial"].T
    fig, axes = plt.subplots(1, 2, figsize=(15, 7.2))
    for ax, key, title in [(axes[0], "leiden_before_fix", "before the fix"),
                           (axes[1], "leiden", f"after: FIX = {FIX!r}")]:
        cats = adata.obs[key].astype("category")
        cm = plt.get_cmap("tab20")
        ax.scatter(xs, ys, s=1.2, linewidths=0, rasterized=True,
                   c=[cm(c % 20) for c in cats.cat.codes])
        ax.set_aspect("equal"); ax.invert_yaxis()
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{title}  ({cats.nunique()} clusters)")
    plt.tight_layout(); plt.show()

Compare this with the same three panels before the fix. What you want to see: the
cluster panel still shows clean, separated groups, while the `total_counts` and
`cell_area` panels no longer align with cluster boundaries.

If the clusters *also* collapsed, you over-corrected — the fix removed biology along
with the depth. Step back to a gentler option.

**Then say what you did.** "Counts were normalised per cell area and log1p
transformed; the first ten principal components showed maximum |r| = 0.31 with total
counts" is a methods sentence a reviewer can evaluate. "Data were normalised using
standard procedures" is not.

**Check the second and third panels before you look at anything else.** If a cluster
lights up on `total_counts` or `cell_area` alone, it is a technical cluster —
depth or segmentation size — not a cell type. That happens far more often in Xenium
than in scRNA-seq because the dynamic range of counts per cell is so compressed.